# CKD evidence landscape

This dashboard-ready notebook presents three separate, public-observed aggregates: MEPS HC-243 2022 person-year utilization/expenditure estimates, a bounded CMS Medicare Part D 2024 therapy dictionary, and a fully paginated ClinicalTrials.gov CKD registry snapshot. Each section retains its own population, time, grain, denominator, and limitations. No person, beneficiary, provider, or study identifier is joined across sources.

The fixture section at the end is explicitly `fixture_only` for offline contract demonstrations. It cannot overwrite or be substituted for the official aggregate artifacts.

In [1]:
from pathlib import Path
import hashlib
import json
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
def load_verified(stem):
    artifact_path = ROOT / 'data' / 'processed' / f'{stem}.json'
    checksum_path = ROOT / 'data' / 'processed' / f'{stem}.sha256'
    manifest_path = ROOT / 'data' / 'manifests' / f'{stem}.json'
    digest = hashlib.sha256(artifact_path.read_bytes()).hexdigest()
    assert digest in checksum_path.read_text(encoding='utf-8')
    artifact = json.loads(artifact_path.read_text(encoding='utf-8'))
    manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
    assert artifact['retrieved_at'] == manifest['retrieved_at']
    return artifact, manifest

meps, meps_manifest = load_verified('meps_hc243_2022_landscape')
partd, partd_manifest = load_verified('partd_2024_ckd_therapy_landscape')
trials, trials_manifest = load_verified('clinicaltrials_ckd_landscape')
pd.DataFrame([{'source': x['source'], 'release': x['release'], 'evidence_type': x['evidence_type'], 'retrieved_at': x['retrieved_at']} for x in (meps, partd, trials)])

,source,release,evidence_type,retrieved_at
0,MEPS / AHRQ,HC-243-2022,public_observed,2026-09-11T22:44:45.239679Z
1,CMS Medicare Part D Prescribers by Provider an...,2024,public_observed,2026-09-11T21:38:23.635394Z
2,ClinicalTrials.gov,api-v2,public_observed,2026-09-11T22:20:34.443896Z


## MEPS: survey-weighted utilization and expenditures

These estimates use `DIABW22F` with `VARSTR`/`VARPSU` for the documented DCS diabetes-related kidney-problem proxy. `DSKIDN53` is not a confirmed CKD diagnosis.

In [2]:
pd.DataFrame(meps['estimates'])[['metric', 'unit', 'point', 'standard_error', 'ci_low', 'ci_high', 'denominator', 'sum_weights']]

,metric,unit,point,standard_error,ci_low,ci_high,denominator,sum_weights
0,proxy_positive_prevalence,proportion,0.111775,0.011113,0.089844,0.133707,1063,2.804286e+07
1,office_visits,visits per person-year,12.247932,0.558512,11.145731,13.350132,1063,2.804286e+07
2,outpatient_visits,visits per person-year,1.945267,0.238634,1.474333,2.416202,1063,2.804286e+07
3,emergency_visits,visits per person-year,0.318131,0.026575,0.265687,0.370576,1063,2.804286e+07
4,inpatient_stays,stays per person-year,0.160981,0.015628,0.130141,0.191821,1063,2.804286e+07
5,prescription_medicines,medicines per person-year,32.239379,1.158404,29.953319,34.525440,1063,2.804286e+07
6,total_expenditure_usd,USD per person-year,16746.487652,768.247523,15230.384046,18262.591258,1063,2.804286e+07


In [3]:
pd.Series(meps['denominators'], name='MEPS exact record denominators').to_frame()

,MEPS exact record denominators
all_person_records,22431
dcs_eligible_records,1185
design_eligible_person_records,21747
diabetes_reported_records,1077
proxy_complete_records,1063
proxy_missing_records,14
proxy_negative_records,921
proxy_positive_records,142


## CMS Part D: bounded therapy dictionary

The table is a selected, complete-for-the-dictionary provider-drug-state slice. CMS excludes providers with fewer than 11 total claims; absent rows are not zeros.

In [4]:
pd.DataFrame(partd['dictionary'])[['generic_name', 'class']]
pd.DataFrame(partd['aggregates']).head(20)

,brand_name,generic_name,provider_count,provider_state,total_30day_fills,total_claims,total_drug_cost_usd
0,Dapagliflozin,Dapagliflozin Propanediol,9,AK,361.0,175,175931.23
1,Dapagliflozin,Dapagliflozin Propanediol,140,AL,3545.4,3454,1205183.16
2,Dapagliflozin,Dapagliflozin Propanediol,56,AR,1172.0,1138,433127.17
3,Dapagliflozin,Dapagliflozin Propanediol,74,AZ,1641.0,1279,758427.19
4,Dapagliflozin,Dapagliflozin Propanediol,1422,CA,52172.4,44475,24034423.15
5,Dapagliflozin,Dapagliflozin Propanediol,67,CO,1466.9,1423,618579.05
6,Dapagliflozin,Dapagliflozin Propanediol,98,CT,2205.8,1857,836448.64
7,Dapagliflozin,Dapagliflozin Propanediol,2,DC,26.0,26,13595.98
8,Dapagliflozin,Dapagliflozin Propanediol,15,DE,345.0,307,152875.50
9,Dapagliflozin,Dapagliflozin Propanediol,438,FL,19826.9,16825,7291057.05


In [5]:
pd.DataFrame(partd['pagination']['generic_pages'])

,generic,offsets,page_checksums,pages,rows_retrieved
0,Empagliflozin,"[0, 5000, 10000, 15000, 20000, 25000, 30000, 3...","[{'bytes': 2859550, 'offset': 0, 'rows': 5000,...",40,198275
1,Dapagliflozin Propanediol,"[0, 5000, 10000, 15000, 20000, 25000, 30000, 3...","[{'bytes': 2904117, 'offset': 0, 'rows': 5000,...",24,118837
2,Finerenone,"[0, 5000]","[{'bytes': 2823853, 'offset': 0, 'rows': 5000,...",2,8474


## ClinicalTrials.gov: registry landscape

The exact CKD query is fully paginated. Status, phase/type, sponsor, geography, and update-year tables summarize registered studies only. Missing enrollment remains null in status/phase rows and is reported separately.

In [6]:
print(trials['query'])
pd.Series(trials['counts'], name='ClinicalTrials.gov reconciliation').to_frame()

AREA[ConditionSearch]("Chronic Kidney Disease")


,ClinicalTrials.gov reconciliation
api_total_count,3706
dimension_denominators,"{'enrollment_reported': 3675, 'intervention_av..."
missing_optional_fields,"{'enrollment': 31, 'intervention': 501, 'locat..."
quarantine_count,0
quarantine_reasons,{}
retrieved_row_count,3706
studies_with_missing_enrollment,31
studies_with_reported_enrollment,3675
unique_nct_ids,3706
unique_valid_nct_id_count,3706


In [7]:
status = pd.DataFrame(trials['status'])
intervention = pd.DataFrame(trials['intervention']).sort_values('study_count', ascending=False).head(15)
phase_or_type = pd.DataFrame(trials['phase_or_type'])
sponsor = pd.DataFrame(trials['sponsor']).sort_values('study_count', ascending=False).head(15)
geography = pd.DataFrame(trials['geography']).sort_values('study_count', ascending=False).head(15)
change_over_time = pd.DataFrame(trials['change_over_time'])
display(status); display(phase_or_type); display(intervention); display(sponsor); display(geography); display(change_over_time)

,available_denominator,dimension,evidence_type,grain,overall_status,source,studies_with_reported_enrollment,study_count,study_share,total_reported_enrollment
0,3706,overall_status,public_observed,registered clinical study grouped by overall_s...,ACTIVE_NOT_RECRUITING,ClinicalTrials.gov,118,118,0.031840,507811.0
1,3706,overall_status,public_observed,registered clinical study grouped by overall_s...,APPROVED_FOR_MARKETING,ClinicalTrials.gov,0,2,0.000540,NaN
2,3706,overall_status,public_observed,registered clinical study grouped by overall_s...,AVAILABLE,ClinicalTrials.gov,0,1,0.000270,NaN
3,3706,overall_status,public_observed,registered clinical study grouped by overall_s...,COMPLETED,ClinicalTrials.gov,1995,2011,0.542634,27007246.0
4,3706,overall_status,public_observed,registered clinical study grouped by overall_s...,ENROLLING_BY_INVITATION,ClinicalTrials.gov,39,39,0.010523,1145535.0
5,3706,overall_status,public_observed,registered clinical study grouped by overall_s...,NOT_YET_RECRUITING,ClinicalTrials.gov,214,214,0.057744,314490.0
6,3706,overall_status,public_observed,registered clinical study grouped by overall_s...,NO_LONGER_AVAILABLE,ClinicalTrials.gov,0,2,0.000540,NaN
7,3706,overall_status,public_observed,registered clinical study grouped by overall_s...,RECRUITING,ClinicalTrials.gov,419,419,0.113060,794879.0
8,3706,overall_status,public_observed,registered clinical study grouped by overall_s...,SUSPENDED,ClinicalTrials.gov,12,12,0.003238,131793.0
9,3706,overall_status,public_observed,registered clinical study grouped by overall_s...,TERMINATED,ClinicalTrials.gov,204,208,0.056125,52890.0


,available_denominator,dimension,evidence_type,grain,source,studies_with_reported_enrollment,study_count,study_share,study_type,total_reported_enrollment,phase
0,3706,study_type,public_observed,registered clinical study grouped by study_type,ClinicalTrials.gov,0,5,0.001349,EXPANDED_ACCESS,NaN,NaN
1,3706,study_type,public_observed,registered clinical study grouped by study_type,ClinicalTrials.gov,2635,2657,0.716945,INTERVENTIONAL,1270311.0,NaN
2,3706,study_type,public_observed,registered clinical study grouped by study_type,ClinicalTrials.gov,1040,1044,0.281705,OBSERVATIONAL,31312199.0,NaN
3,3706,phase,public_observed,registered clinical study grouped by phase,ClinicalTrials.gov,23,23,0.006206,NaN,1596.0,EARLY_PHASE1
4,3706,phase,public_observed,registered clinical study grouped by phase,ClinicalTrials.gov,2169,2189,0.590664,NaN,32101874.0,NA
5,3706,phase,public_observed,registered clinical study grouped by phase,ClinicalTrials.gov,189,190,0.051268,NaN,6189.0,PHASE1
6,3706,phase,public_observed,registered clinical study grouped by phase,ClinicalTrials.gov,55,56,0.015111,NaN,4442.0,PHASE1|PHASE2
7,3706,phase,public_observed,registered clinical study grouped by phase,ClinicalTrials.gov,365,365,0.098489,NaN,43406.0,PHASE2
8,3706,phase,public_observed,registered clinical study grouped by phase,ClinicalTrials.gov,45,45,0.012142,NaN,14904.0,PHASE2|PHASE3
9,3706,phase,public_observed,registered clinical study grouped by phase,ClinicalTrials.gov,445,448,0.120885,NaN,266093.0,PHASE3


,available_denominator,evidence_type,grain,intervention,share_of_studies_with_intervention,source,study_count
2320,3205,public_observed,registered study grouped by intervention name,Placebo,0.115133,ClinicalTrials.gov,369
845,3205,public_observed,registered study grouped by intervention name,Darbepoetin alfa,0.011232,ClinicalTrials.gov,36
3680,3205,public_observed,registered study grouped by intervention name,placebo,0.009048,ClinicalTrials.gov,29
2655,3205,public_observed,registered study grouped by intervention name,Roxadustat,0.007488,ClinicalTrials.gov,24
2052,3205,public_observed,registered study grouped by intervention name,No intervention,0.007176,ClinicalTrials.gov,23
2225,3205,public_observed,registered study grouped by intervention name,Paricalcitol,0.005928,ClinicalTrials.gov,19
813,3205,public_observed,registered study grouped by intervention name,Dapagliflozin,0.005928,ClinicalTrials.gov,19
3137,3205,public_observed,registered study grouped by intervention name,Usual Care,0.005928,ClinicalTrials.gov,19
605,3205,public_observed,registered study grouped by intervention name,Cholecalciferol,0.004680,ClinicalTrials.gov,15
1144,3205,public_observed,registered study grouped by intervention name,Exercise,0.004680,ClinicalTrials.gov,15


,evidence_type,grain,share_of_registered_studies,source,sponsor,study_count
85,public_observed,registered study grouped by lead sponsor,0.017809,ClinicalTrials.gov,Assiut University,66
94,public_observed,registered study grouped by lead sponsor,0.017539,ClinicalTrials.gov,AstraZeneca,65
56,public_observed,registered study grouped by lead sponsor,0.016730,ClinicalTrials.gov,Amgen,62
125,public_observed,registered study grouped by lead sponsor,0.014841,ClinicalTrials.gov,Bayer,55
439,public_observed,registered study grouped by lead sponsor,0.014301,ClinicalTrials.gov,Hoffmann-La Roche,53
769,public_observed,registered study grouped by lead sponsor,0.007285,ClinicalTrials.gov,Novo Nordisk A/S,27
1277,public_observed,registered study grouped by lead sponsor,0.006746,ClinicalTrials.gov,VA Office of Research and Development,25
35,public_observed,registered study grouped by lead sponsor,0.006206,ClinicalTrials.gov,Akebia Therapeutics,23
1191,public_observed,registered study grouped by lead sponsor,0.006206,ClinicalTrials.gov,"University of Colorado, Denver",23
92,public_observed,registered study grouped by lead sponsor,0.005936,ClinicalTrials.gov,Astellas Pharma Inc,22


,available_denominator,country,evidence_type,grain,share_of_studies_with_location,source,study_count
105,3300,United States,public_observed,registered study-country mention,0.329697,ClinicalTrials.gov,1088
30,3300,France,public_observed,registered study-country mention,0.093030,ClinicalTrials.gov,307
17,3300,China,public_observed,registered study-country mention,0.092424,ClinicalTrials.gov,305
32,3300,Germany,public_observed,registered study-country mention,0.082727,ClinicalTrials.gov,273
104,3300,United Kingdom,public_observed,registered study-country mention,0.078788,ClinicalTrials.gov,260
15,3300,Canada,public_observed,registered study-country mention,0.070303,ClinicalTrials.gov,232
91,3300,Spain,public_observed,registered study-country mention,0.066364,ClinicalTrials.gov,219
46,3300,Italy,public_observed,registered study-country mention,0.063636,ClinicalTrials.gov,210
78,3300,Poland,public_observed,registered study-country mention,0.052121,ClinicalTrials.gov,172
96,3300,Taiwan,public_observed,registered study-country mention,0.049394,ClinicalTrials.gov,163


,evidence_type,grain,source,study_count,update_year
0,public_observed,registered study grouped by last update year,ClinicalTrials.gov,6,2005
1,public_observed,registered study grouped by last update year,ClinicalTrials.gov,19,2006
2,public_observed,registered study grouped by last update year,ClinicalTrials.gov,21,2007
3,public_observed,registered study grouped by last update year,ClinicalTrials.gov,36,2008
4,public_observed,registered study grouped by last update year,ClinicalTrials.gov,37,2009
5,public_observed,registered study grouped by last update year,ClinicalTrials.gov,48,2010
6,public_observed,registered study grouped by last update year,ClinicalTrials.gov,86,2011
7,public_observed,registered study grouped by last update year,ClinicalTrials.gov,80,2012
8,public_observed,registered study grouped by last update year,ClinicalTrials.gov,84,2013
9,public_observed,registered study grouped by last update year,ClinicalTrials.gov,131,2014


## Decision boundaries

- MEPS describes a weighted survey population and its person-year utilization/expenditures; it is not provider prescribing data.
- Part D describes selected provider-drug aggregates; it is not an indication, adherence, outcomes, or beneficiary dataset.
- ClinicalTrials.gov describes registry submissions; recruitment status and phase are not proof of active enrollment, efficacy, or market share.
- These evidence cards may be displayed together for context, but no combined patient-level score or causal/commercial claim is valid without compatible populations, time windows, and analytic denominators.

## Offline fixture-only contract check

CI fixtures are intentionally separate and are not used to populate the official outputs above.

In [8]:
from ckd_intelligence.ingestion.trials import ingest_trials
fixture = ingest_trials(ROOT / 'tests' / 'fixtures' / 'ingestion' / 'clinicaltrials.json', cache_dir=ROOT / 'data' / 'raw' / 'task5-fixture-cache')
fixture_rows = [{**dict(row), 'evidence_type': 'fixture_only'} for row in fixture.valid]
assert {row['evidence_type'] for row in fixture_rows} == {'fixture_only'}
print({'fixture_only_notebook_namespace': True, 'records': len(fixture_rows), 'official_artifacts_unchanged': True})

{'fixture_only_notebook_namespace': True, 'records': 1, 'official_artifacts_unchanged': True}
